## New notebook to read output of MD simulation, using Adios2 library

### part 1: **print information summary**

- Reading one of the sample output files (.bp), step by step.  
- Printing available variables and attributes, as well as their structure.

In [ ]:
import numpy as np
from adios2 import Stream
import os

def read_adios_output(outputDir, index, print_summary=True):
    attributes = {}
    variables = {}
    
    filename = os.path.join(outputDir, f"run_{index}.bp")

    with Stream(filename, "r") as s:
        for i, _ in enumerate(s.steps()):
            if i == 0:
                for attr in s.available_attributes():
                    attributes[attr] = s.read_attribute(attr)
            for var in s.available_variables():
                if var not in variables:
                    variables[var] = []
                variables[var].append(s.read(var))

    for var in variables:
        variables[var] = np.array(variables[var])

    if print_summary:
        print("Attributes Summary:")
        print("-------------------")
        for idx, (name, value) in enumerate(attributes.items(), start=1):
            print(f"{idx}. {name:<40} {value}")
        print("\nVariables Summary:")
        print("------------------")
        for idx, (name, arr) in enumerate(variables.items(), start=1):
            shape = arr.shape
            if arr.ndim == 1:
                description = shape[0]
            else:
                description = f"{shape[0]} * {list(shape[1:])}"
            print(f"{idx}. {name:<35} {description}")

    return {"attributes": attributes, "variables": variables}

outputDir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/"
result = read_adios_output(outputDir, 0)


---

### Part 2: **Read and Plot Neighbor Counts**

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader
from scipy.signal import savgol_filter

def read_variable(output_dir, variable_name):
    pattern = re.compile("run_([0-9]+).bp")
    run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
    
    if not run_numbers:
        raise ValueError(f"No run files found in {output_dir}")
    
    run_numbers.sort()

    variable_data = []
    temperatures = []
    run_indeces = []
    
    for run_num in run_numbers:
        file_path = os.path.join(output_dir, f"run_{run_num}.bp")
        with FileReader(file_path) as reader:
            if variable_name in reader.available_variables():
                var_info = reader.available_variables()[variable_name]
                steps = int(var_info.get("AvailableStepsCount", 1))
                
                data = reader.read(variable_name, step_selection=[steps - 1, 1])
                data = data.flatten()
                variable_data.append(data)
                
                temp_label = reader.read_attribute("temperature")
                temp_label = temp_label.flatten()
                temperatures.append(temp_label)
                
                run_index = reader.read_attribute("runIndex")
                run_index = run_index.flatten()
                run_indeces.append(run_index)
            else:
                raise ValueError(f"Variable {variable_name} not found in {file_path}")
            
    return np.array(variable_data), np.array(temperatures), np.array(run_indeces)

def plot_neighbors(neighbors_array, smoothing=True, window_length=10, polyorder=3):
    variable_data = neighbors_array[0]
    temperatures = neighbors_array[1]
    run_indeces = neighbors_array[2]
    
    fig, ax = plt.subplots(figsize=(10, 6))

    for i in range(variable_data.shape[1]):
        y = variable_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        ax.plot(y, label=f"Shell {i}")

    # total_points = len(temperatures)
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    ax.set_xlabel("Simulation Procedure, temperature")
    
    ax.set_xticks(tick_indices)
    ax.set_xticklabels([f"T={temperatures[i][0]:.2f}" for i in tick_indices], rotation=45)
    # ax.set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    
    ax.set_ylabel("Number of Neighbors")
    ax.set_title("Neighbor Analysis (Last Step)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()
    
output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/"
neighbors = read_variable(output_dir, "number of neighbors")
plot_neighbors(neighbors)
